# f-I Curve Analysis — Mitral & Tufted Cells

Characterizes **spike frequency vs injected current (f-I)** for all five Birgiolas 2020
mitral cell (MC) and tufted cell (TC) models.

Two protocols are compared:
- **Step protocol** — one independent simulation per current level; cell reinitializes each time.
- **Ramp protocol** — single simulation with a staircase current waveform; spike times binned per epoch.

All currents are in **nA** (NEURON native units).


## Setup

In [ ]:
import sys, os
# ensure repo root is on the path when running from notebooks/
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from fi_curve_utils import (
    run_fi_steps,
    run_fi_ramp,
    compute_fi_slope,
    run_cell_type_fi,
    plot_fi_curves,
    plot_voltage_traces,
    build_slope_table,
    _get_cell_class,
)

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 9})

### Protocol Parameters

Edit these values to change the current range, step size, or epoch duration
for **both** protocols below.


In [ ]:
# Current range — all values in nA
I_START_nA   = 0.0    # first step
I_STOP_nA    = 0.6    # last step (inclusive)
I_STEP_nA    = 0.05   # increment between steps

# Timing
STEP_DUR_MS  = 500.0  # duration of each constant-current epoch (ms)
DELAY_MS     = 200.0  # silent period before current onset (ms)
TAIL_MS      = 50.0   # silent period after current offset (ms)

# Integration
DT_MS        = 0.1    # fixed timestep (ms)
CELSIUS      = 35.0   # temperature (°C)
THRESHOLD_MV = -20.0  # APCount spike-detection threshold (mV)

# Shared kwargs forwarded to both protocol functions
PROTOCOL_KW = dict(
    i_start_nA   = I_START_nA,
    i_stop_nA    = I_STOP_nA,
    i_step_nA    = I_STEP_nA,
    step_dur_ms  = STEP_DUR_MS,
    delay_ms     = DELAY_MS,
    tail_ms      = TAIL_MS,
    dt           = DT_MS,
    celsius      = CELSIUS,
    threshold_mV = THRESHOLD_MV,
)

---
## 1. Step Protocol (independent runs)

Each current level gets its own simulation. `h.run()` reinitializes the cell
from `h.v_init` before each step, so there is no history between levels.


In [ ]:
print('Running MC step protocol...')
mc_steps = run_cell_type_fi('MC', protocol='steps', **PROTOCOL_KW)
print()
print('Running TC step protocol...')
tc_steps = run_cell_type_fi('TC', protocol='steps', **PROTOCOL_KW)

In [ ]:
fig_steps = plot_fi_curves(
    {'MC': mc_steps, 'TC': tc_steps},
    protocol_label='step protocol',
)
plt.show()

### Step Protocol — Slope Summary


In [ ]:
df_steps = build_slope_table({'MC': mc_steps, 'TC': tc_steps})
df_steps.style \
    .format({
        'Slope_Hz_per_nA': '{:.1f}',
        'Intercept_Hz':    '{:.1f}',
        'R2':              '{:.4f}',
        'Max_Freq_Hz':     '{:.1f}',
        'Rheobase_nA':     '{:.2f}',
    }) \
    .background_gradient(subset=['Slope_Hz_per_nA'], cmap='viridis')

---
## 2. Ramp Protocol (staircase in a single simulation)

All current levels run in one `h.run()` call. A `h.Vector` drives the
IClamp amplitude at each timestep, stepping up every `STEP_DUR_MS` ms.
Spike times are recorded and binned per epoch.

This protocol captures any history effects (e.g. adaptation state carried
across epochs) that the step protocol masks by re-initializing each time.


In [ ]:
print('Running MC ramp protocol...')
mc_ramp = run_cell_type_fi('MC', protocol='ramp', **PROTOCOL_KW)
print()
print('Running TC ramp protocol...')
tc_ramp = run_cell_type_fi('TC', protocol='ramp', **PROTOCOL_KW)

In [ ]:
fig_ramp = plot_fi_curves(
    {'MC': mc_ramp, 'TC': tc_ramp},
    protocol_label='ramp protocol',
)
plt.show()

### Ramp Protocol — Slope Summary


In [ ]:
df_ramp = build_slope_table({'MC': mc_ramp, 'TC': tc_ramp})
df_ramp.style \
    .format({
        'Slope_Hz_per_nA': '{:.1f}',
        'Intercept_Hz':    '{:.1f}',
        'R2':              '{:.4f}',
        'Max_Freq_Hz':     '{:.1f}',
        'Rheobase_nA':     '{:.2f}',
    }) \
    .background_gradient(subset=['Slope_Hz_per_nA'], cmap='viridis')

---
## 3. Step vs Ramp Comparison

Overlays both protocols on the same axes to reveal history-dependent differences.


In [ ]:
def _plot_protocol_comparison(step_results, ramp_results, cell_type):
    colors = plt.get_cmap('tab10').colors
    fig, ax = plt.subplots(figsize=(7, 5))

    for idx, name in enumerate(step_results):
        c = colors[idx % len(colors)]
        s = step_results[name]
        r = ramp_results[name]
        ax.plot(s['currents_nA'], s['freqs_hz'], 'o-',  color=c, lw=1.5,
                label=f'{name} step')
        ax.plot(r['currents_nA'], r['freqs_hz'], 's--', color=c, lw=1.5,
                alpha=0.7, label=f'{name} ramp')

    ax.set_title(f'{cell_type} — Step (o-) vs Ramp (s--)', fontsize=11)
    ax.set_xlabel('Injected Current (nA)')
    ax.set_ylabel('Firing Frequency (Hz)')
    ax.legend(fontsize=7, ncol=2, loc='upper left')
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    return fig

_plot_protocol_comparison(mc_steps, mc_ramp, 'MC')
plt.show()
_plot_protocol_comparison(tc_steps, tc_ramp, 'TC')
plt.show()

### Δ Slope: Ramp − Step

Negative values mean the ramp slope is shallower than the step slope,
consistent with spike-frequency adaptation carrying over between epochs.


In [ ]:
df_cmp_rows = []
for cell_type, step_res, ramp_res in [
    ('MC', mc_steps, mc_ramp),
    ('TC', tc_steps, tc_ramp),
]:
    for name in step_res:
        s_slope = step_res[name]['slope']
        r_slope = ramp_res[name]['slope']
        df_cmp_rows.append(dict(
            Cell=name, Type=cell_type,
            Step_Hz_per_nA=s_slope,
            Ramp_Hz_per_nA=r_slope,
            Delta_Hz_per_nA=r_slope - s_slope,
        ))

df_cmp = pd.DataFrame(df_cmp_rows).set_index('Cell')
df_cmp.style \
    .format({
        'Step_Hz_per_nA':  '{:.1f}',
        'Ramp_Hz_per_nA':  '{:.1f}',
        'Delta_Hz_per_nA': '{:+.1f}',
    }) \
    .background_gradient(subset=['Delta_Hz_per_nA'], cmap='RdBu', vmin=-100, vmax=100)

---
## 4. Voltage Traces

Record soma membrane potential across a set of current amplitudes to
visually verify spike patterns and check for adaptation or bursting.

Edit `TRACE_CELLS` and `TRACE_CURRENTS_nA` to inspect different models.


In [ ]:
# Which cells to show traces for, and which current levels
TRACE_CELLS       = ['MC1', 'TC1']
TRACE_CURRENTS_nA = [0.1, 0.2, 0.3, 0.4]  # nA

for cell_name in TRACE_CELLS:
    cls = _get_cell_class(cell_name)
    fig = plot_voltage_traces(
        cls,
        currents_nA  = TRACE_CURRENTS_nA,
        step_dur_ms  = STEP_DUR_MS,
        delay_ms     = DELAY_MS,
        tail_ms      = TAIL_MS,
        dt           = DT_MS,
        celsius      = CELSIUS,
    )
    plt.show()